  # 01 — Brent crude oil in euros
  Download daily Brent prices (USD/barrel) and the EUR/USD exchange rate from FRED.

In [1]:
  import pandas as pd
  import matplotlib.pyplot as plt
  from pathlib import Path

  RAW = Path("../data/raw")
  PROCESSED = Path("../data/processed")
  FIGURES = Path("../figures")

  for folder in [RAW, PROCESSED, FIGURES]:
      folder.mkdir(parents=True, exist_ok=True)

In [2]:
  FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"

  series = {
      "DCOILBRENTEU": "brent_usd",
      "DEXUSEU": "usd_per_eur",
  }

  for code, name in series.items():
      df = pd.read_csv(FRED_URL.format(code), index_col=0, parse_dates=True, na_values=".")
      df.columns = [name]
      df.to_csv(RAW / f"{name}.csv")
      print(f"{name}: {len(df)} rows, from {df.index.min().date()} to {df.index.max().date()}")

brent_usd: 10270 rows, from 1987-05-20 to 2026-09-29
usd_per_eur: 7235 rows, from 1999-01-04 to 2026-09-25


## Brent in euros per litre, weekly average

In [3]:
brent = pd.read_csv(RAW / "brent_usd.csv", index_col=0, parse_dates=True)
fx = pd.read_csv(RAW / "usd_per_eur.csv", index_col=0, parse_dates=True)

df = brent.join(fx, how="inner").dropna()
df = df.loc["2019-01-01":]

LITRES_PER_BARREL = 158.987
df["brent_eur_bbl"] = df["brent_usd"] / df["usd_per_eur"]
df["brent_eur_litre"] = df["brent_eur_bbl"] / LITRES_PER_BARREL

weekly = df.resample("W").mean()
weekly.to_csv(PROCESSED / "brent_eur_weekly.csv")
weekly.tail()

,brent_usd,usd_per_eur,brent_eur_bbl,brent_eur_litre
observation_date,,,,
2026-08-30,89.7300,1.164820,77.033619,0.484528
2026-09-06,99.0925,1.160650,85.374572,0.536991
2026-09-13,113.6675,1.162175,97.807516,0.615192
2026-09-20,124.1460,1.151440,107.811481,0.678115
2026-09-27,117.0840,1.141360,102.587463,0.645257
